#### Create Bronze Schema (Delta Tables 

In [0]:
use catalog project;


CREATE TABLE IF NOT EXISTS bronze.orders_raw (
  order_id      BIGINT,
  order_ts      STRING,
  customer_id   BIGINT,
  product_id    STRING,
  category      STRING,
  qty           INT,
  price         DECIMAL(10,2),
  status        STRING,
  ingestion_ts  TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS bronze.customers_raw (
  customer_id    BIGINT,
  customer_name  STRING,
  email          STRING,
  city           STRING,
  state          STRING,
  country        STRING,
  segment        STRING,
  update_ts      STRING,
  ingestion_ts   TIMESTAMP
) USING DELTA;


In [0]:
select * from bronze.orders_raw;

order_id,order_ts,customer_id,product_id,category,qty,price,status,ingestion_ts


In [0]:
%python
orders_raw_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = false)



#### Batch Ingestion into Bronze

In [0]:
%python
from pyspark.sql.functions import current_timestamp

orders_raw_df = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv("abfss://data@trainingbr.dfs.core.windows.net/raw/orders")
         .withColumn("ingestion_ts", current_timestamp())
)

orders_raw_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("project.bronze.orders_raw1")

customers_raw_df = (
    spark.read
         .option("header", "true")
         .option("inferSchema", "true")
         .csv("abfss://data@trainingbr.dfs.core.windows.net/raw/customers")
         .withColumn("ingestion_ts", current_timestamp())
)

customers_raw_df.write.format("delta") \
    .mode("append") \
    .saveAsTable("project.bronze.customers_raw1")


In [0]:
insert into project.bronze.orders_raw  select * from  project.bronze.orders_raw1;
insert into project.bronze.customers_raw  select * from  project.bronze.customers_raw1;


num_affected_rows,num_inserted_rows
6,6


In [0]:
select * from project.bronze.orders_raw;

order_id,order_ts,customer_id,product_id,category,qty,price,status,ingestion_ts
1001,2025-01-01 10:15:00,1,P01,Electronics,1,500.00,PAID,2025-12-13T06:54:38.157693Z
1002,2025-01-01 10:18:00,2,P02,Books,2,250.00,PAID,2025-12-13T06:54:38.157693Z
1003,2025-01-01 10:20:00,1,P03,Electronics,1,1200.00,CANCELLED,2025-12-13T06:54:38.157693Z
1004,2025-01-01 10:22:00,3,P02,Books,1,125.00,PAID,2025-12-13T06:54:38.157693Z
1005,2025-01-01 10:25:00,4,P04,Toys,2,300.00,PAID,2025-12-13T06:54:38.157693Z
1002,2025-01-01 10:18:00,2,P02,Books,2,250.00,PAID,2025-12-13T06:54:38.157693Z
1006,2025-01-02 09:00:00,2,P01,Electronics,1,500.00,PAID,2025-12-13T06:54:38.157693Z
1007,2025-01-02 09:05:00,5,P05,Fashion,3,700.00,PAID,2025-12-13T06:54:38.157693Z
1008,2025-01-02 09:10:00,999,P06,Electronics,1,800.00,PAID,2025-12-13T06:54:38.157693Z


In [0]:
select * from bronze.customers_raw;

customer_id,customer_name,email,city,state,country,segment,update_ts,ingestion_ts
1,Amit Kumar,amit@example.com,Chennai,TN,India,RET,2025-01-01 00:00:00,2025-12-13T06:54:43.786621Z
2,Latha Rao,latha@example.com,Bangalore,KA,India,RET,2025-01-01 00:00:00,2025-12-13T06:54:43.786621Z
3,Rahul Jain,rahul@example.com,Hyderabad,TS,India,RET,2025-01-01 00:00:00,2025-12-13T06:54:43.786621Z
4,Meena Iyer,meena@example.com,Mumbai,MH,India,ENT,2025-01-01 00:00:00,2025-12-13T06:54:43.786621Z
2,Latha R,latha.r@example.com,Bangalore,KA,India,ENT,2025-01-02 00:00:00,2025-12-13T06:54:43.786621Z
5,Arjun Dev,arjun@example.com,Delhi,DL,India,RET,2025-01-02 00:00:00,2025-12-13T06:54:43.786621Z
